<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_audio0426.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# Cell 1: 纯净底座与系统音频依赖
# ==========================================
import os

print("🧹 正在彻底清空旧环境，防止残留干扰...")
!rm -rf /content/ComfyUI

print("⏳ 安装系统级底层音频依赖 (FFmpeg & libsndfile1)...")
!apt update -qq
!apt -y install ffmpeg libsndfile1 -qq

print("🔄 下载全新 ComfyUI 官方源码...")
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git

print("📦 安装核心底座依赖...")
%cd /content/ComfyUI
!pip install -q -r requirements.txt
print("✅ Cell 1 完成！")

🧹 正在彻底清空旧环境，防止残留干扰...
⏳ 安装系统级底层音频依赖 (FFmpeg & libsndfile1)...
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.
🔄 下载全新 ComfyUI 官方源码...
/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 36728, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 36728 (delta 115), reused 85 (delta 85), pack-reused 36587 (from 2)
Receiving objects: 100% (36728/36728), 85.34 MiB | 15.70 MiB/s, done.
Resolving deltas: 100% (24599/24599), done.
📦 安装核心底座依赖...
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 123.

In [2]:
# ==========================================
# Cell 2: 音频核心插件群 & 强制稳定依赖环境
# ==========================================
%cd /content/ComfyUI/custom_nodes

print("⏳ [1/3] 下载 FishAudioS2 (核心语音)...")
!git clone https://github.com/Saganaki22/ComfyUI-FishAudioS2.git

print("⏳ [2/3] 下载 VideoHelperSuite (用来加载和保存生成的音频)...")
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

print("⏳ [3/3] 下载 KJNodes (提供额外的音频控制节点)...")
!git clone https://github.com/kijai/ComfyUI-KJNodes.git

print("📦 正在静默安装所有音频插件的依赖...")
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-FishAudioS2/requirements.txt
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-KJNodes/requirements.txt

print("💉 注入环境稳定剂：打死不兼容的底层库...")
# 降级 Numpy，修复 Triton 报错，锁定 CUDA 12 的 bitsandbytes，补齐音频处理包
!pip install -q "numpy<2.0.0" triton "bitsandbytes==0.43.3" resampy soundfile librosa pydub huggingface_hub

%cd /content/ComfyUI
print("✅ Cell 2 完成！音频专属插件群已就绪！")

/content/ComfyUI/custom_nodes
⏳ [1/3] 下载 FishAudioS2 (核心语音)...
Cloning into 'ComfyUI-FishAudioS2'...
remote: Enumerating objects: 898, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 898 (delta 61), reused 71 (delta 36), pack-reused 787 (from 1)
Receiving objects: 100% (898/898), 22.90 MiB | 19.24 MiB/s, done.
Resolving deltas: 100% (390/390), done.
⏳ [2/3] 下载 VideoHelperSuite (用来加载和保存生成的音频)...
Cloning into 'ComfyUI-VideoHelperSuite'...
remote: Enumerating objects: 3355, done.
remote: Counting objects: 100% (1601/1601), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 3355 (delta 1463), reused 1235 (delta 1232), pack-reused 1754 (from 3)
Receiving objects: 100% (3355/3355), 793.38 KiB | 1014.00 KiB/s, done.
Resolving deltas: 100% (1978/1978), done.
⏳ [3/3] 下载 KJNodes (提供额外的音频控制节点)...
Cloning into 'ComfyUI-KJNodes'...
remote: Enumerating objects: 4850, done.
remote: Counting objects: 100% (1733/17

In [4]:
# ==========================================
# Cell 3: 预下载 Fish-Speech 核心大模型 (强制静默版)
# ==========================================
import os
print("📥 正在开启多线程满速下载 Fish-Speech-1.5 官方大模型...")
print("⏳ 这个过程需要几分钟，请耐心等待进度条走完...")

# 1. 强制静默更新依赖，并开启高速传输
!pip install -q -U huggingface_hub hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 2. 使用纯 Python API 下载，彻底封杀命令行 [Y/n] 弹窗
from huggingface_hub import snapshot_download

try:
    path = snapshot_download(repo_id="fishaudio/fish-speech-1.5")
    print(f"\n✅ Cell 3 完成！大模型已死死锁在本地缓存中。")
    print("🚀 现在去运行 Cell 4 启动吧！")
except Exception as e:
    print(f"\n❌ 下载出现意外: {e}")

📥 正在开启多线程满速下载 Fish-Speech-1.5 官方大模型...
⏳ 这个过程需要几分钟，请耐心等待进度条走完...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 34.1 MB/s eta 0:00:00


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Cell 3 完成！大模型已死死锁在本地缓存中。
🚀 现在去运行 Cell 4 启动吧！


In [5]:
# ==========================================
# Cell 4: 启动专属音频服务器
# ==========================================
%cd /content/ComfyUI
import threading
import time
import urllib.request

# 1. 获取公网密码
print("👇 请复制下方出现的 IP 密码，稍后打开网页时需要填入（如提示）：")
password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"👉 密码: {password} 👈")

# 2. 安装并拉起穿透
!npm install -g localtunnel -q > /dev/null 2>&1

# 3. 核心启动命令
def run_comfyui():
    !python main.py --dont-print-server

# 4. 后台拉起 ComfyUI
threading.Thread(target=run_comfyui, daemon=True).start()

# 给它几秒钟的热身时间
time.sleep(5)

print("🌐 正在生成访问链接，请点击下方带有 'loca.lt' 的 URL 打开界面...")
!lt --port 8188

/content/ComfyUI
👇 请复制下方出现的 IP 密码，稍后打开网页时需要填入（如提示）：
👉 密码: 34.143.163.120 👈
setup plugin alembic.autogenerate.schemas
setup plugin alembic.autogenerate.tables
setup plugin alembic.autogenerate.types
setup plugin alembic.autogenerate.constraints
setup plugin alembic.autogenerate.defaults
setup plugin alembic.autogenerate.comments
🌐 正在生成访问链接，请点击下方带有 'loca.lt' 的 URL 打开界面...
your url is: https://warm-teams-drum.loca.lt
Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Found comfy_kitchen backend eager: {'availa